In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
import matplotlib.pyplot as plt

In [3]:
# Fetching data for Apple
ticker = 'AAPL'
data = yf.download(ticker, start='2010-01-01', end='2020-12-31')
data.shape

C:\Users\15887\AppData\Local\Temp\ipykernel_12568\1341501773.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start='2010-01-01', end='2020-12-31')
[*********************100%***********************]  1 of 1 completed


(2768, 5)

In [4]:
df = data[['Close']]
df.head()

Price,Close
Ticker,AAPL
Date,
2010-01-04,6.424605
2010-01-05,6.435712
2010-01-06,6.333344
2010-01-07,6.321636
2010-01-08,6.363664


In [5]:
df.columns

MultiIndex([('Close', 'AAPL')],
           names=['Price', 'Ticker'])

In [6]:
## Preprocess the data
# 1. Scale between 0-1
scaler = MinMaxScaler(feature_range=(0,1))
scaler_data = scaler.fit_transform(np.array(df).reshape(-1,1))
scaler_data

array([[0.00517357],
       [0.00526074],
       [0.00445738],
       ...,
       [1.        ],
       [0.9860826 ],
       [0.97728907]], shape=(2768, 1))

In [7]:
print(type(df))
type(scaler_data)

<class 'pandas.core.frame.DataFrame'>


numpy.ndarray

In [8]:
# 2. Train and test data
train_size = int(len(scaler_data)* 0.8)
train_data = scaler_data[:train_size] # train data from 0 to 80% of data using index
test_data = scaler_data[train_size:] # test data from 80% of index to 100%



In [9]:

'''

Below function will create data in given manner, this will use steps 
and devide data into train and test from all the available time series data

Train data : 100, 120, 110, 134, 150,144

   X-Train     y_train
f1,  f2,  f3     o/p
100, 120, 110    134
120, 110, 134    150
110, 134, 150    144
'''

def create_dataset(data ,  time_step =1):
    print('creating dataset')
    X, y = [], []
    for i in range(len(data)-time_step - 1):
        a = data[i:(i + time_step), 0]
        # print(f'a: {a}')
        X.append(a)
        y.append(data[i+time_step,0])
        # print(f'y: {y}')
    return np.array(X), np.array(y)

In [10]:
time_step = 60
X_train, y_train = create_dataset(train_data, time_step=time_step)
X_test, y_test = create_dataset(test_data, time_step=time_step)

creating dataset
creating dataset


In [11]:
# print(train_data[60:63,0])
# train_data[:100]

In [12]:
# y_train

In [13]:
# 3. Reshaping for  input to be [Samples, time_steps, features] for LSTM
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

In [14]:
## Build LSTM Model
model = Sequential()
model.add(LSTM(units=50, return_sequences=True, input_shape=(X_train.shape[1], 1)))
model.add(Dropout(0.2))
model.add(LSTM(units=50, return_sequences=True))
model.add(Dropout(0.2))
model.add(Dense(    units=1))

C:\Users\15887\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [15]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 60, 50)         │        10,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 50)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 60, 50)         │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 60, 50)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 60, 1)          │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,651 (119.73 KB)

 Trainable params: 30,651 (119.73 KB)

 Non-trainable params: 0 (0.00 B)

In [102]:
model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=100,batch_size=64,verbose=1)

Epoch 1/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 3s 48ms/step - loss: 0.0096 - val_loss: 0.1593
Epoch 2/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - loss: 0.0077 - val_loss: 0.1983
Epoch 3/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 3s 74ms/step - loss: 0.0076 - val_loss: 0.1956
Epoch 4/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 68ms/step - loss: 0.0075 - val_loss: 0.1915
Epoch 5/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - loss: 0.0074 - val_loss: 0.2013
Epoch 6/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - loss: 0.0075 - val_loss: 0.1894
Epoch 7/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step - loss: 0.0074 - val_loss: 0.1966
Epoch 8/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 51ms/step - loss: 0.0074 - val_loss: 0.1951
Epoch 9/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 51ms/step - loss: 0.0075 - val_loss: 0.1855
Epoch 10/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step - loss: 0.0074 - val_loss: 0.1853
Epoch 11/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step - loss: 0.0074 - val_loss: 0.1952
Epoch 12/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step

In [103]:
# train the model
# model.fit(X_train, y_train, epochs=100, batch_size=32)

In [104]:
# model prediction
train_predict = model.predict(X_train)
test_predict = model.predict(X_test)

68/68 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step


In [82]:
# train_predict

In [106]:
## Inverse scaling to get actual predict values
train_predict = train_predict.reshape(-1, 1)
test_predict = test_predict.reshape(-1,1)
train_predict1= scaler.inverse_transform(train_predict)
test_predict1 = scaler.inverse_transform(test_predict)

In [87]:
X_train1= scaler.inverse_transform(X_train.reshape(-1,1))

In [107]:
train_predict_plot = np.empty_like(scaler_data)
train_predict_plot[:, :] = np.nan
train_predict_plot[time_step:len(train_predict1)+time_step+1, :] = train_predict1

ValueError: could not broadcast input array from shape (129180,1) into shape (2708,1)

In [81]:
# train_predict

In [80]:
# train_predict1

In [79]:
# df